# Baseline model development

# Day 3 - Explore delay propagation and establish baselines

This notebook analyzes linked Southwest (WN) aircraft-flight pairs from January-April 2026. The target is whether the current flight departs at least 15 minutes late (`DEP_DEL15`). It explores delay patterns by hour, weekday, origin airport, previous-flight arrival delay, and scheduled turnaround slack, then compares a majority-class baseline with an initial logistic-regression model.

**Evaluation design:** January-February are used for the initial baseline, January for candidate fitting, and February for model/threshold selection. March is a previously inspected comparison. The selected model is then refit on January-March and evaluated on April as the fresh final holdout. All imputing, scaling, and encoding are fitted on training data only.

**Interpretation limit:** this is an initial historical benchmark, not an operational dispatch recommendation. The available sample covers four months and one carrier.


In [1]:

from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Find the workspace root from the notebook working directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "processed").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data" / "processed").is_dir():
    raise FileNotFoundError("Open this notebook from the flight project folder.")
PAIR_PATH = ROOT / "data" / "processed" / "wn_aircraft_pairs_2026-01_to_2026-04.csv"
REPORT_DIR = ROOT / "reports" / "metrics"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)



## 1. Load the linked-pair data and inspect the target

Each row represents a current flight and the immediately preceding flight assigned to the same aircraft. The pair construction rules and raw-data quality profile are documented in the Day 2 notebook. Current-flight actual delay is used to create the target, but will not be included among the predictors.


In [2]:

if not PAIR_PATH.is_file():
    raise FileNotFoundError(f"Run notebooks/02_data_preprocessing.ipynb first; missing {PAIR_PATH}")

pairs = pd.read_csv(PAIR_PATH, low_memory=False, parse_dates=["FLIGHT_DATE"])
required = {
    "FLIGHT_DATE", "DEP_DEL15", "DEP_DELAY_MIN", "PREV_ARR_DELAY_MIN",
    "SCHEDULED_TURNAROUND_SLACK_MIN", "ORIGIN", "DEST",
    "DEPARTURE_HOUR_LOCAL", "DAY_OF_WEEK", "MONTH",
}
missing = sorted(required - set(pairs.columns))
if missing:
    raise ValueError(f"Pair file is missing required fields: {missing}")
pairs["FLIGHT_DATE"] = pd.to_datetime(pairs["FLIGHT_DATE"], errors="coerce")
pairs = pairs.dropna(subset=["FLIGHT_DATE", "DEP_DEL15"]).copy()
pairs["DEP_DEL15"] = pairs["DEP_DEL15"].astype(int)

print("Pair table shape:", pairs.shape)
print("Date range:", pairs["FLIGHT_DATE"].min().date(), "to", pairs["FLIGHT_DATE"].max().date())
print("Target counts:")
print(pairs["DEP_DEL15"].value_counts().rename(index={0: "under 15 min", 1: "15+ min"}))
print("Overall 15+ minute departure-delay rate:", round(pairs["DEP_DEL15"].mean(), 4))
print("Missing values in candidate fields:")
print(pairs[list(required)].isna().sum().sort_values(ascending=False).to_string())


Pair table shape: (423660, 22)
Date range: 2026-01-01 to 2026-04-30
Target counts:
DEP_DEL15
under 15 min    321558
15+ min         102102
Name: count, dtype: int64
Overall 15+ minute departure-delay rate: 0.241
Missing values in candidate fields:
DEST                              0
DEPARTURE_HOUR_LOCAL              0
FLIGHT_DATE                       0
DEP_DELAY_MIN                     0
ORIGIN                            0
SCHEDULED_TURNAROUND_SLACK_MIN    0
DEP_DEL15                         0
DAY_OF_WEEK                       0
MONTH                             0
PREV_ARR_DELAY_MIN                0


## 1. Chronological holdout and leakage-safe baseline models

## 5. Chronological holdout and leakage-safe baseline models

January and February train the initial baseline; March is an intermediate comparison. April is reserved as the fresh final holdout. The simple baseline always predicts the most common class in training. Logistic regression uses numeric inbound delay and scheduled slack plus categorical route, hour, and weekday. Month is excluded because March is the held-out test month and has no category observed during training.

`DEP_DELAY_MIN`, actual departure/arrival times, and current-flight post-outcome fields are deliberately excluded from predictors. Imputation, scaling, and one-hot encoding are fitted only when the model pipeline is trained on January-February. Logistic regression uses balanced class weights because the positive class is a minority; this changes the precision/recall trade-off, so both are reported.


In [6]:

cutoff = pd.Timestamp("2026-03-01")
train = pairs.loc[pairs["FLIGHT_DATE"] < cutoff].copy()
march_end = pd.Timestamp("2026-04-01")
test = pairs.loc[(pairs["FLIGHT_DATE"] >= cutoff) & (pairs["FLIGHT_DATE"] < march_end)].copy()
if train.empty or test.empty:
    raise ValueError("Initial comparison requires Jan-Feb training rows and March comparison rows.")

target = "DEP_DEL15"
numeric_features = ["PREV_ARR_DELAY_MIN", "SCHEDULED_TURNAROUND_SLACK_MIN"]
categorical_features = ["ORIGIN", "DEST", "DEPARTURE_HOUR_LOCAL", "DAY_OF_WEEK"]
feature_columns = numeric_features + categorical_features
X_train, y_train = train[feature_columns], train[target]
X_test, y_test = test[feature_columns], test[target]
print("Training dates:", train["FLIGHT_DATE"].min().date(), "to", train["FLIGHT_DATE"].max().date(), "rows:", len(train))
print("Test dates:", test["FLIGHT_DATE"].min().date(), "to", test["FLIGHT_DATE"].max().date(), "rows:", len(test))
print("Training positive rate:", round(float(y_train.mean()), 4), "Test positive rate:", round(float(y_test.mean()), 4))
print("Predictors:", feature_columns)

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=25))]), categorical_features),
    ],
    remainder="drop",
)
models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Balanced logistic regression": Pipeline([
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(class_weight="balanced", max_iter=500, solver="liblinear")),
    ]),
}

def score_model(name, model):
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        probability = model.predict_proba(X_test)[:, 1]
    else:
        probability = predicted.astype(float)
    return {
        "MODEL": name,
        "TEST_ROWS": int(len(y_test)),
        "TEST_POSITIVE_RATE": float(y_test.mean()),
        "ACCURACY": float(accuracy_score(y_test, predicted)),
        "PRECISION": float(precision_score(y_test, predicted, zero_division=0)),
        "RECALL": float(recall_score(y_test, predicted, zero_division=0)),
        "F1": float(f1_score(y_test, predicted, zero_division=0)),
        "ROC_AUC": float(roc_auc_score(y_test, probability)) if len(np.unique(y_test)) == 2 else None,
        "PR_AUC_AVERAGE_PRECISION": float(average_precision_score(y_test, probability)),
    }

results = pd.DataFrame([score_model(name, model) for name, model in models.items()])
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Training dates: 2026-01-01 to 2026-02-28 rows: 194870
Test dates: 2026-03-01 to 2026-03-31 rows: 114706
Training positive rate: 0.1963 Test positive rate: 0.3022
Predictors: ['PREV_ARR_DELAY_MIN', 'SCHEDULED_TURNAROUND_SLACK_MIN', 'ORIGIN', 'DEST', 'DEPARTURE_HOUR_LOCAL', 'DAY_OF_WEEK']
                       MODEL  TEST_ROWS  TEST_POSITIVE_RATE  ACCURACY  PRECISION  RECALL     F1  ROC_AUC  PR_AUC_AVERAGE_PRECISION
           Majority baseline     114706              0.3022    0.6978     0.0000  0.0000 0.0000   0.5000                    0.3022
Balanced logistic regression     114706              0.3022    0.7767     0.6071  0.7399 0.6670   0.8374                    0.7066


## 2. Save the first results and confusion matrix

## 6. Save the first results and confusion matrix

Accuracy alone can hide poor detection of delayed flights. Review precision, recall, F1, ROC-AUC, and average precision together. The confusion matrix below uses rows for actual outcomes and columns for predictions. Threshold tuning and final model selection belong to Day 4 and must use validation data, not this test set.


In [7]:

results_path = REPORT_DIR / "day3_baseline_metrics.csv"
results.to_csv(results_path, index=False)
logistic = models["Balanced logistic regression"]
logistic_prediction = logistic.predict(X_test)
cm = confusion_matrix(y_test, logistic_prediction, labels=[0, 1])
cm_table = pd.DataFrame(cm, index=["Actual under 15 min", "Actual 15+ min"], columns=["Predicted under 15 min", "Predicted 15+ min"])
cm_table.to_csv(REPORT_DIR / "day3_logistic_confusion_matrix.csv")
print("Logistic-regression confusion matrix:")
print(cm_table.to_string())

run_summary = {
    "dataset": str(PAIR_PATH.relative_to(ROOT)),
    "target": "DEP_DEL15 (current departure delay >= 15 minutes)",
    "train_period": f"{train['FLIGHT_DATE'].min().date()} to {train['FLIGHT_DATE'].max().date()}",
    "test_period": f"{test['FLIGHT_DATE'].min().date()} to {test['FLIGHT_DATE'].max().date()}",
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "feature_columns": feature_columns,
    "leakage_excluded": ["DEP_DELAY_MIN", "ACTUAL_DEP_UTC", "ACTUAL_ARR_UTC", "PREV_ACTUAL_ARR_UTC"],
    "results": results.to_dict(orient="records"),
    "outputs": [
        str(p.relative_to(ROOT)) for p in [
            FIGURE_DIR / "delay_distributions.png",
            FIGURE_DIR / "delay_rates_by_schedule_and_origin.png",
            FIGURE_DIR / "propagation_by_inbound_delay_and_slack.png",
            REPORT_DIR / "day3_delay_rate_by_hour.csv",
            REPORT_DIR / "day3_delay_rate_by_weekday.csv",
            REPORT_DIR / "day3_delay_rate_by_origin.csv",
            REPORT_DIR / "day3_propagation_by_inbound_delay.csv",
            REPORT_DIR / "day3_propagation_by_scheduled_slack.csv",
            results_path,
            REPORT_DIR / "day3_logistic_confusion_matrix.csv",
            REPORT_DIR / "day3_delay_quantiles.csv",
            REPORT_DIR / "day3_run_summary.json",
        ]
    ],
}
(REPORT_DIR / "day3_run_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")


Logistic-regression confusion matrix:
                     Predicted under 15 min  Predicted 15+ min
Actual under 15 min                   63442              16597
Actual 15+ min                         9018              25649


1888

## Baseline model handoff and limitations

## Day 3 handoff and limitations

- The February-selected model and threshold are refit on January-March and evaluated on the newly added April holdout in the final section. March was already viewed during baseline exploration; do not use April to make further tuning choices.
- This is association and prediction on retrospective BTS records; it does not prove causal propagation or identify operational actions.
- The downloaded BTS fields are a selected subset, and this analysis covers four months of one carrier. Weather, airport capacity, crew legality, aircraft swaps, and real-time schedule updates are not fully represented.
- Review outliers and class-specific errors before reporting conclusions. Do not remove extreme delay values solely because they are large.
